In [1]:

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
!nvidia-smi
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import  sklearn
from sklearn.model_selection import train_test_split
#import ray
import os
import math
import matplotlib.pyplot as plt
import tensorflow.keras.optimizers.schedules as schedules
import pickle

Tue Nov 28 08:36:52 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  On   | 00000000:05:00.0 Off |                  N/A |
|  0%   30C    P8    13W / 198W |      1MiB /  8192MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  On   | 00000000:06:00.0 Off |                  N/A |
| 27%   

2023-11-28 08:36:53.462572: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-28 08:36:53.618960: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-11-28 08:36:55.157809: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-11-28 08:36:55.157918: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or 

In [ ]:


gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

originaldir = os.getcwd()
originaldir

"""# Data processing

For the homework, you should download the dataset with the provided link. The provided dataset has already partitioned the data by clients.
"""

os.chdir('/nfs/home/dem1110/Assignment 3/Part 2/')

train_data = 'train_data.npy'
train_data_arr = np.load(train_data, allow_pickle=True)
test_data = 'test_data.npy'
test_data_arr = np.load(test_data, allow_pickle=True)

os.chdir(originaldir)
os.getcwd()

"""# Define model"""

## regular model
'''
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Flatten(input_shape=(28, 28)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return model
'''
## adding leanring rate.
def get_model(lr):


    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(62, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

"""## Adding Noise"""

#np.where(b, b+a, b)

#b = 0.1

###defining functions
def test(test_data_arr, weights, batch_size, lr):
        model = get_model(lr)  # Construct the model
        model.set_weights(weights)  # Set model weights with the latest parameters

        # Assuming test_data is a tuple of test_images and test_labels
        test_images = np.array(test_data_arr[0]['images'])
        test_labels = np.array(test_data_arr[0]['labels'])



        # Normalize the input data
        #test_images = np.array(test_images)
        #test_images= test_images / 255.0

        test_images_flat = test_images.reshape(test_images.shape[0], -1)

        test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
        batch_size = batch_size  # Set your desired batch size
        # Evaluate the model using the dataset
        batched_test_dataset = test_dataset.batch(batch_size)

        test_loss, test_accuracy = model.evaluate(batched_test_dataset)


        print(f"Test Accuracy: {test_accuracy:.4f}")

        return test_loss, test_accuracy
    
class Client:
    def __init__(self, i, batch_size, initial_learning_rate, E):
        # Create model
        self.model = get_model(initial_learning_rate)
        self.training_data = all_train[i]
        self.validation_data = all_val[i]
        self.batch_size = batch_size
        self.E = E

    def train(self, global_weights):
        self.model.set_weights(global_weights)

        self.model.fit(self.training_data.batch(self.batch_size), epochs=self.E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)

        return self.model.get_weights()

    def evaluate(self):
        #gives scalars of loss and accuracy at the end of training
        train_loss, train_accuracy = self.model.evaluate(self.training_data.batch(self.batch_size), verbose=0)
        val_loss, val_accuracy = self.model.evaluate(self.validation_data.batch(self.batch_size), verbose=0)
        return train_loss, train_accuracy, val_loss, val_accuracy

sample_nbs = []
for i in range(100):

    sample_nbs.append(len(train_data_arr[i]['labels']))

def weighted_average(sampled_clients, metric):
    # TODO: Aggregate client losses or accuracies by taking a weighted average.
    #       Weights are proportional to the number of samples at each client
    sample_size_selected = [sample_nbs[i] for i in sampled_clients]
    #sorted list so order should match
    agg_train_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_accuracy'])) / sum(sample_size_selected)
    agg_train_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_loss'])) / sum(sample_size_selected)
    agg_val_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_accuracy'])) / sum(sample_size_selected)
    agg_val_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_loss'])) / sum(sample_size_selected)


    return agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss

"""# Aggregation of client updates"""

def fedAvg(sampled_clients, client_weights):
    # Get the size of samples for each client
    sample_size_selected = [sample_nbs[i] for i in sampled_clients]
    #print("shape of local weights: ", len(client_weights[0]), client_weights[0][0].shape)
    list_dim = len(client_weights[0])
    np_dim = client_weights[0][0].shape
    total = sum(sample_size_selected)

    avg_weights = []
    for k in range(list_dim):
        avg_weights.append(np.zeros(client_weights[0][k].shape))

    for i in range (len(sampled_clients)):
        scalar = sample_size_selected[i]/total
        for j in range(0, list_dim):
            this_numpy = np.multiply(client_weights[i][j], scalar)
            avg_weights[j] = np.add(avg_weights[j],this_numpy)
    #print("shape of avg weights: ", len(avg_weights), avg_weights[0].shape)
    return avg_weights

def plots_train(run, t, client, E, batch_size,  train_loss, train_accuracy, val_loss, val_accuracy):
    plt.figure(figsize=(12, 4))
    #plt.subplot(1, 2, 1)
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Rounds')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(run + '_train_loss_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()
    # Plot training accuracy and validation accuracy
    plt.figure(figsize=(12, 4))
    plt.plot(train_accuracy, label='Training Accuracy')
    plt.plot(val_accuracy, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Rounds')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(run + '_train_accuracy_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()
    
    

b_options = [0.08, 0.09, 0.1, 0.2, 0.3,0.4,  0.5,0.6, 0.7,  0.8,0.9, 1,1.1, 1.2, 1.3, 1.4, 1.5,  2, 3, 4, 5, 0.05,0.06, 0.07]

plot_metrics = {}
plot_metrics["b"] = []
plot_metrics["last train acc"] = []
plot_metrics["last validation acc"] = []
plot_metrics["test acc"] = []

for b in b_options:
    for i in range (0, 100):
        nb_images = len(train_data_arr[i]["images"])
        #print(i)
        for j in range(0, nb_images):
            c = np.add(train_data_arr[i]["images"][j], np.random.laplace(loc=0.0, scale=b, size=(28,28)))
            #threshold = 1
            #c = np.where(b<threshold, b+a, b)
            train_data_arr[i]["images"][j] = c

    nb_images = len(test_data_arr[0]["images"])
    for j in range(0, nb_images):
        c = np.add(test_data_arr[0]["images"][j], np.random.laplace(loc=0.0, scale=b, size=(28,28)))
        #threshold = 1
        #c = np.where(b<threshold, b+a, b)
        test_data_arr[0]["images"][j] = c

    plt.imshow(train_data_arr[67]["images"][56], interpolation="None")
    plt.savefig("b="+str(b)+"_client_68, 57th image_.png")
    plt.close()

    """## Splitting Data"""

    all_train = []
    all_val = []
    x_train_all = []
    y_train_all = []
    x_val_all =[]
    y_val_all =[]

    for i in range(100):

        data = train_data_arr[i]
        # Split the data into training and validation sets
        images_train, images_val, labels_train, labels_val = train_test_split(
            data['images'],
            data['labels'],
            test_size=0.2,
            random_state=42)
        train_dataset = tf.data.Dataset.from_tensor_slices((images_train, labels_train))
        val_dataset = tf.data.Dataset.from_tensor_slices((images_val, labels_val))
        all_train.append(train_dataset)
        all_val.append(val_dataset)

    """# Client class"""

    #@ray.remote(num_cpus=1, num_gpus=0)


    """# Aggregated metrics and losses
    Aggregate client metrics and losses for (federated) evaluation metrics.
    """



    """# Launch the learning"""

    run = "part2_b="+str(b)
    Agg_metric = {}
    Agg_metric['train_loss'] = []
    Agg_metric['train_accuracy'] =[]
    Agg_metric['val_loss'] =[]
    Agg_metric['val_accuracy'] =[]

    # Initialize the global model weights
    batch_size = 32
    # Initialize the global model weights with an initial learning rate
    initial_learning_rate = 0.00001
    global_model = get_model(initial_learning_rate)

    #global_model = get_model()
    users = np.arange(100)
    # Get data
    #data_by_clients, test_data = partition_data()

    total_rounds = 500

    C = 10   # Only a fraction of C of clients participate in the local training per round
    E = 30     # Number of epochs for each local training

    for round in range(total_rounds):
        # Step 1
        print("communication round: ", round)
        #initiate empty list of model weights
        client_weights =[]
        metric = {}
        metric['train_loss'] = []
        metric['train_accuracy'] =[]
        metric['val_loss'] =[]
        metric['val_accuracy'] =[]
        # TODO: Sample C fraction of clients
        sampled_clients = np.random.choice(users, C, replace=False)
        sampled_clients = sorted(sampled_clients)
        # TODO: Get the local data for the selected clients
        for i in sampled_clients:
            #print("client: ", i)
            #perform local training
            #define instance of client:
            current_client = Client(i, batch_size, initial_learning_rate, E)

            #save local weights

            local_weights = current_client.train(global_model.get_weights())
            #print(local_weights[0].shape)
            client_weights.append(local_weights)
            #save local results

            train_loss, train_accuracy, val_loss, val_accuracy = current_client.evaluate()
            metric['train_loss'].append(train_loss)
            metric['train_accuracy'].append(train_accuracy)
            metric['val_loss'].append(val_loss)
            metric['val_accuracy'].append(val_accuracy)
            #print("train_loss : ", train_loss)
            #print("train_accuracy: ", train_accuracy)


        # Step 3
        # TODO: Aggregate the losses and metrics, keep track of the metrics
        agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss = weighted_average(sampled_clients, metric)
        Agg_metric['train_loss'].append(agg_train_loss)
        Agg_metric['train_accuracy'].append(agg_train_acc)
        Agg_metric['val_loss'].append(agg_val_loss)
        Agg_metric['val_accuracy'].append(agg_val_acc)
        print("aggregated validation accuracy: ",Agg_metric['val_accuracy'][round])
        # Step 4
        # TODO: Aggregate the client updates
        weighted_avg_weights = fedAvg(sampled_clients, client_weights)

        # Step 5
        # Update the global model
        global_model.set_weights(weighted_avg_weights)

        #if round%50==0 and round!=0:


        # Specify the file path
    file_path = 'Agg_metric_'+run+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.pickle'

    # Save the dictionary to a pickle file
    with open(file_path, 'wb') as file:
        pickle.dump(Agg_metric, file)

    print(f'Dictionary saved to {file_path}')

    global_model.save('global_model_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.h5')
    plots_train(run, round, C, E, batch_size,  Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])

    """# Create plots"""

    

    plots_train(run, total_rounds, C, E, batch_size, Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
    # Evaluate on test data after training
    test_loss, test_accuracy = test(test_data_arr, global_model.get_weights(), batch_size, initial_learning_rate)

    print("test loss: ", test_loss)
    print("test accuracy: ", test_accuracy)

    test_results = np.array ( [b, Agg_metric['train_accuracy'][-1],Agg_metric['val_accuracy'][-1],  test_loss, test_accuracy])
    
    np.save('test_results_b='+str(b)+'_'+run+'_'+str(total_rounds)+'_'+str(C)+'_'+str(E)+'_'+str(batch_size)+'_.npy', test_results)
    plot_metrics["b"].append(b)
    plot_metrics["last train acc"].append(Agg_metric['train_accuracy'][-1])
    plot_metrics["last validation acc"].append(Agg_metric['val_accuracy'][-1])
    plot_metrics["test acc"].append(test_accuracy)                


file_path2 = 'Plotting metrics for all bs.pickle'

# Save the dictionary to a pickle file
with open(file_path2, 'wb') as file:
    pickle.dump(plot_metrics, file)

print(f'Dictionary saved to {file_path2}')
                       








2023-11-28 08:37:00.197247: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-28 08:37:00.692373: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6553 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:0a:00.0, compute capability: 6.1


communication round:  0
aggregated validation accuracy:  0.046641342423605035
communication round:  1
aggregated validation accuracy:  0.04766214838616177
communication round:  2
aggregated validation accuracy:  0.06923337934940858
communication round:  3
aggregated validation accuracy:  0.056084574873999385
communication round:  4
aggregated validation accuracy:  0.07694549068886165
communication round:  5
aggregated validation accuracy:  0.0868845356338972
communication round:  6
aggregated validation accuracy:  0.07760235626305222
communication round:  7
aggregated validation accuracy:  0.08541000627251844
communication round:  8
aggregated validation accuracy:  0.07183023185777943
communication round:  9
aggregated validation accuracy:  0.09982027853737999
communication round:  10
aggregated validation accuracy:  0.1006087042338612
communication round:  11
aggregated validation accuracy:  0.12966014125413255
communication round:  12
aggregated validation accuracy:  0.11566461311743

2023-11-28 09:12:27.247898: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


aggregated validation accuracy:  0.6124259289365465
communication round:  209
aggregated validation accuracy:  0.5730926228067779
communication round:  210
aggregated validation accuracy:  0.5493578728446811
communication round:  211
aggregated validation accuracy:  0.5389316967902782
communication round:  212
aggregated validation accuracy:  0.5595305902418185
communication round:  213
aggregated validation accuracy:  0.5954723220612408
communication round:  214
aggregated validation accuracy:  0.5648772936228131
communication round:  215
aggregated validation accuracy:  0.6033493523006734
communication round:  216
aggregated validation accuracy:  0.6113995436259678
communication round:  217
aggregated validation accuracy:  0.5783751418552538
communication round:  218
aggregated validation accuracy:  0.5776761188687258
communication round:  219
aggregated validation accuracy:  0.5342127040242465
communication round:  220
aggregated validation accuracy:  0.6119343468169666
communicatio

2023-11-28 10:07:08.552695: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


...

aggregated validation accuracy:  0.30655427290353293
communication round:  285
aggregated validation accuracy:  0.267547362553815
communication round:  286
aggregated validation accuracy:  0.29407230592900657
communication round:  287
aggregated validation accuracy:  0.30453798587964276
communication round:  288
aggregated validation accuracy:  0.36448150958834574
communication round:  289
aggregated validation accuracy:  0.2893691017119468
communication round:  290
aggregated validation accuracy:  0.31515229721726523
communication round:  291
aggregated validation accuracy:  0.2955202099549201
communication round:  292
aggregated validation accuracy:  0.30396240366743277
communication round:  293
aggregated validation accuracy:  0.3154311244722557
communication round:  294
aggregated validation accuracy:  0.3114243000481099
communication round:  295
aggregated validation accuracy:  0.3152697211446788
communication round:  296
aggregated validation accuracy:  0.29221442188996727
commun

2023-11-28 15:33:56.579389: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


aggregated validation accuracy:  0.3181755455706886
communication round:  335
aggregated validation accuracy:  0.32416386714681206
communication round:  336
aggregated validation accuracy:  0.3074990203769035
communication round:  337
aggregated validation accuracy:  0.3661474534098983
communication round:  338
aggregated validation accuracy:  0.2959925808264855
communication round:  339
aggregated validation accuracy:  0.34527213689930075
communication round:  340
aggregated validation accuracy:  0.33730125532093297
communication round:  341
aggregated validation accuracy:  0.3390340888804763
communication round:  342
aggregated validation accuracy:  0.34503427522339947
communication round:  343
aggregated validation accuracy:  0.32564603727130237
communication round:  344
aggregated validation accuracy:  0.3343221016357966
communication round:  345
aggregated validation accuracy:  0.30613303255093727
communication round:  346
aggregated validation accuracy:  0.33952280490213865
commu

2023-11-28 16:01:17.233062: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


aggregated validation accuracy:  0.32686735861749355
communication round:  485
aggregated validation accuracy:  0.3562742916239871
communication round:  486
aggregated validation accuracy:  0.329434321313546
communication round:  487
aggregated validation accuracy:  0.32671031516331894
communication round:  488
aggregated validation accuracy:  0.3414556390631505
communication round:  489
aggregated validation accuracy:  0.3121894043693363
communication round:  490
aggregated validation accuracy:  0.33394272337696757
communication round:  491
aggregated validation accuracy:  0.3664875779383929
communication round:  492
aggregated validation accuracy:  0.29290434121687275
communication round:  493
aggregated validation accuracy:  0.3097811236135299
communication round:  494
aggregated validation accuracy:  0.30546975813388827
communication round:  495
aggregated validation accuracy:  0.37088053099506973
communication round:  496
aggregated validation accuracy:  0.3114564680622856
communi

## Plotting across the scale. 

In [ ]:


#os.chdir('/nfs/home/dem1110/Assignment 3/Part 2/running multiple bs/')

In [ ]:

'''
file_path = "Plotting metrics for all bs_withmore.pickle"

with open(file_path, 'rb') as file:
    # Load the dictionary from the pickle file
    plot_metrics = pickle.load(file)
'''

plt.plot(plot_metrics["b"], plot_metrics["last train acc"], label='Training Accuracy')
plt.plot(plot_metrics["b"], plot_metrics["last validation acc"], label='Validation Accuracy')
plt.plot(plot_metrics["b"], plot_metrics["test acc"], label='Test Accuracy')


plt.xlabel('Scale (b) ')
plt.ylabel('Accuracy')
plt.title('Run 2 - Accuracies across different scales')
plt.legend()

# Display the plot

plt.savefig("run2 - accuracies across the scale.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
x = np.arange(len(plot_metrics["b"]))
plt.bar(x, plot_metrics["test acc"], color='magenta', width=0.5, edgecolor='black')
plt.xlabel('Scale (b) Values ')
plt.ylabel('Test Accuracy')
plt.title('Bar Plot - Test Accuracy Across Different Scales')

#plt.xticks(rotation=45)  # Rotating x-axis labels for better readability
plt.xticks(x + 0.1, plot_metrics["b"]) 
plt.tight_layout()

plt.savefig("run2 - bar plot Test accuracy the scale.png")
plt.show()

In [ ]:
bar_width = 0.2  # Width of each bar
plt.close()
x = np.arange(len(plot_metrics["b"]))

# Plotting grouped bar plots
#plt.figure(figsize=(10, 6))
plt.figure(figsize=(10, 6))
plt.bar(x, plot_metrics["last train acc"], width=bar_width, color='skyblue', label='Training Accuracy')
plt.bar(x + bar_width, plot_metrics["last validation acc"], width=bar_width, color='orange', label='Validation Accuracy')
plt.bar(x + 2 * bar_width, plot_metrics["test acc"], width=bar_width, color='green', label='Test Accuracy')

plt.xlabel('Scale Values (b) ')
plt.ylabel('Accuracy')
plt.title('Accuracies for Each Scale')
plt.xticks(x + bar_width, plot_metrics["b"])  # Set x-axis labels to category names

plt.legend()
#plt.tight_layout()

plt.savefig('run2_bar_plot_accuracies_along_the_scale.png', bbox_inches='tight')
plt.show()